In [1]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-5-nano")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config = {
    "configurable": {
        "thread_id": "1",
    },
}

In [2]:
class State(MessagesState):
    pass


graph_builder = StateGraph(State)

In [3]:
def chatbot(state: State):
    response = llm.invoke(state["messages"])  # langchain api
    return {"messages": [response]}

In [4]:
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn),
)

In [6]:
result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                # "content": "Let's have a short, one-sentence conversation. Hello, My name is Jimin.",
                "content": "I'm going to Osaka soon.",
            },
        ]
    },
    config=config,
)

In [7]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Let's have a short, one-sentence conversation. Hello, My name is Jimin.
================================== Ai Message ==================================

Hi Jimin, nice to meet you!
================================ Human Message =================================

I'm going to Osaka soon.
================================== Ai Message ==================================

That sounds exciting—Osaka is amazing for food and sights; do you have any plans yet?


In [ ]:
state_history = graph.get_state_history(config)  # generator

for state_snapshot in list(state_history):
    print(state_snapshot.next)
    print(state_snapshot.values["messages"])
    print("=========\n")

()
[HumanMessage(content="Let's have a short, one-sentence conversation. Hello, My name is Jimin.", additional_kwargs={}, response_metadata={}, id='011f068f-ce0f-4faf-af11-333aecd49626'), AIMessage(content='Hi Jimin, nice to meet you!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 338, 'prompt_tokens': 24, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DOhnz7rfuqAQs2RmybXRF8F8growF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d392b-4467-78a1-8115-3b8ad566d54a-0', usage_metadata={'input_tokens': 24, 'output_tokens': 338, 'total_tokens': 362, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasonin

In [ ]:
state_history = graph.get_state_history(config)  # generator

to_fork = list(state_history)[-5]
to_fork.values["messages"]

[HumanMessage(content="Let's have a short, one-sentence conversation. Hello, My name is Jimin.", additional_kwargs={}, response_metadata={}, id='011f068f-ce0f-4faf-af11-333aecd49626'),
 AIMessage(content='Hi Jimin, nice to meet you!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 338, 'prompt_tokens': 24, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DOhnz7rfuqAQs2RmybXRF8F8growF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d392b-4467-78a1-8115-3b8ad566d54a-0', usage_metadata={'input_tokens': 24, 'output_tokens': 338, 'total_tokens': 362, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning'

In [ ]:
to_fork.config  # checkpoint

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12b5b4-3237-6406-8003-8ae6fe406519'}}

In [13]:
from langchain_core.messages import HumanMessage

graph.update_state(
    to_fork.config,
    {
        "messages": [
            HumanMessage(
                content="I'm going to Fukuoka soon.",
                id="316af5e0-9cf8-4460-8066-ae7ef90bbeb1",
            )
        ]
    },
)

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12b5d8-5b1c-6df2-8004-ae3214ce96d2'}}

In [14]:
forked_state = graph.get_state_history(
    {
        "configurable": {
            "thread_id": "1",
            "checkpoint_ns": "",
            "checkpoint_id": "1f12b5d8-5b1c-6df2-8004-ae3214ce96d2",
        }
    },
)

list(forked_state)

[StateSnapshot(values={'messages': [HumanMessage(content="Let's have a short, one-sentence conversation. Hello, My name is Jimin.", additional_kwargs={}, response_metadata={}, id='011f068f-ce0f-4faf-af11-333aecd49626'), AIMessage(content='Hi Jimin, nice to meet you!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 338, 'prompt_tokens': 24, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DOhnz7rfuqAQs2RmybXRF8F8growF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d392b-4467-78a1-8115-3b8ad566d54a-0', usage_metadata={'input_tokens': 24, 'output_tokens': 338, 'total_tokens': 362, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_

In [15]:
result = graph.invoke(
    None,
    {
        "configurable": {
            "thread_id": "1",
            "checkpoint_ns": "",
            "checkpoint_id": "1f12b5d8-5b1c-6df2-8004-ae3214ce96d2",
        }
    },
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Let's have a short, one-sentence conversation. Hello, My name is Jimin.
================================== Ai Message ==================================

Hi Jimin, nice to meet you!
================================ Human Message =================================

I'm going to Fukuoka soon.
================================== Ai Message ==================================

Nice—Fukuoka has great food, beaches, and temples; do you have any plans yet or would you like a quick tip?
